# Thermal-Ops with GRPO using TRL

Train an agentic model to manage Data Center Cooling.


In [1]:
!nvidia-smi
!pip install trackio
%pip install --no-cache-dir auto-gptq optimum --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu121/
!pip install ipywidgets

Mon Mar 30 10:37:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Log in to Hugging Face


In [2]:
from huggingface_hub import notebook_login

notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## Define the system prompt


In [3]:
prompt = """You are an expert Data Center Facility Manager with deep knowledge of thermodynamics, power management, and optimal cooling strategies.

Follow these rules to play Thermal-Ops:

1. The target is to maintain 3 server racks within 20.0 to 25.0 Celsius
2. You have a limited number of steps to optimize the temperature while minimizing energy consumption
3. After each step, you receive thermal and cost feedback:
   - GREEN (Optimal): Under 25.0C -> Minimal energy cost applied
   - YELLOW (Warning): Over 25.0C -> Throttling penalty applied
   - RED (Critical): Over 27.0C -> Hardware damage penalty applied
4. All actions must respect physical limits (e.g. max 5000 RPM)
5. You cannot use fans that have experienced a hardware failure
6. Use the tools `set_fan_speed`, `adjust_chiller`, `migrate_workload`, and `wait` to make your moves.
"""



## Define the environment
The `ThermalOpsEnv` class implements all the thermodynamics simulation explicitly inside the environment.


In [4]:
import json
import torch
import optimum

class ThermalOpsEnv:
    def __init__(self):
        self.num_racks = 3
        self.max_steps = 10
        self.w1_energy = 0.5
        self.w2_overheat = 1000.0
        
        self.safe_temp_max = 25.0
        self.critical_temp = 27.0
        
        self.reset()
        
    def reset(self, **kwargs) -> str:
        self.step_count = 0
        self.ambient_temp = 22.0
        self.rack_temps = [22.0, 22.0, 22.0]
        self.power_loads = [10.0, 10.0, 10.0]
        self.fan_rpms = [1000, 1000, 1000]
        self.chiller_setpoint = 15.0
        self.energy_cost = 0.15
        
        self.total_energy_consumed = 0.0
        self.broken_fans = set()
        
        self.reward = 0.0
        self.done = False
        
        return self._get_obs()
        
    def _get_obs(self) -> str:
        obs = {
            "ambient_temp": self.ambient_temp,
            "rack_temps": [round(t, 2) for t in self.rack_temps],
            "power_loads": [round(l, 2) for l in self.power_loads],
            "fan_rpms": self.fan_rpms,
            "chiller_setpoint": round(self.chiller_setpoint, 2),
            "step_count": self.step_count
        }
        return f"Observation: {json.dumps(obs)}"

    def set_fan_speed(self, rack_id: int, rpm: int) -> str:
        """
        Set the fan speed for a specific server rack.

        Args:
            rack_id: The ID of the server rack (0, 1, or 2).
            rpm: The speed to set the fan to in RPM (0 to 5000). High RPM cools faster but uses exponential power.
            
        Returns:
            Status message of the action.
        """
        if self.done: return "Episode over."
        if 0 <= rack_id < self.num_racks and rack_id not in self.broken_fans:
            self.fan_rpms[rack_id] = max(0, min(5000, rpm))
            return f"Fan {rack_id} speed bounded and set to {self.fan_rpms[rack_id]} RPM."
        return "Failed: Invalid rack_id or fan is broken."

    def adjust_chiller(self, chiller_temp: float) -> str:
        """
        Adjust the ambient liquid chiller base temperature setpoint.

        Args:
            chiller_temp: The temperature to set the chiller to (in Celsius).
            
        Returns:
            Status message.
        """
        if self.done: return "Episode over."
        self.chiller_setpoint = max(5.0, min(30.0, float(chiller_temp)))
        return f"Chiller setpoint adjusted to {self.chiller_setpoint}°C."

    def migrate_workload(self, source_rack: int, target_rack: int) -> str:
        """
        Migrate power load (heat generation) from one rack to another.

        Args:
            source_rack: The rack ID to pull workload from.
            target_rack: The rack ID to push workload to.
            
        Returns:
            Status message.
        """
        if self.done: return "Episode over."
        if 0 <= source_rack < self.num_racks and 0 <= target_rack < self.num_racks and source_rack != target_rack:
            load_to_move = self.power_loads[source_rack] * 0.5
            self.power_loads[source_rack] -= load_to_move
            self.power_loads[target_rack] += load_to_move
            return f"Migrated {load_to_move:.2f} workload from rack {source_rack} to rack {target_rack}."
        return "Failed: Invalid source or target rack."
        
    def wait(self) -> str:
        """
        Wait and progress the environment thermodynamics by 1 simulation step.

        Returns:
            The new Observation of the environment after the step.
        """
        if self.done:
            raise ValueError("Environment episode is finished.")
            
        energy_consumed = 0.0
        overheat_penalty_total = 0.0
        
        chiller_delta = max(0, self.ambient_temp - self.chiller_setpoint)
        energy_consumed += 0.5 * (chiller_delta ** 2)
        
        for i in range(self.num_racks):
            heat_generated = 0.1 * self.power_loads[i]
            rpm = self.fan_rpms[i] if i not in self.broken_fans else 0
            cooling_power = (rpm / 1000.0) * 0.5
            chiller_effect = max(0, self.rack_temps[i] - self.chiller_setpoint) * 0.1
            ambient_effect = (self.ambient_temp - self.rack_temps[i]) * 0.05
            
            self.rack_temps[i] += heat_generated - cooling_power - chiller_effect + ambient_effect
            energy_consumed += ((rpm / 1000.0) ** 3) * 0.2
            
            if self.rack_temps[i] > self.safe_temp_max:
                if self.rack_temps[i] > self.critical_temp:
                    overheat_penalty_total += self.w2_overheat
                else:
                    overheat_penalty_total += self.w2_overheat * 0.1 * (self.rack_temps[i] - self.safe_temp_max)
                    
        cost = energy_consumed * self.energy_cost
        self.total_energy_consumed += energy_consumed
        
        # Calculate step reward
        self.reward = -(self.w1_energy * cost) - overheat_penalty_total
        
        self.step_count += 1
        if self.step_count >= self.max_steps:
            self.done = True
            
        return self._get_obs()



## Define the reward function
The reward function retrieves the final `env.reward` score.


In [5]:
def reward_func(environments, **kwargs) -> list[float]:
    return [env.reward for env in environments]
    pass

## Create Dataset for Rollouts


In [6]:
from datasets import Dataset

dataset = Dataset.from_dict({"prompt": [[{"role": "user", "content": prompt}] for _ in range(500)]})


## Set GRPO Config


In [7]:
# 1. Install/Update bitsandbytes and accelerate (required for 4-bit)
%pip install -U bitsandbytes>=0.46.1 accelerate transformers trl datasets

# 2. Re-verify the install
import bitsandbytes as bnb
import torch
print(f"Bitsandbytes version: {bnb.__version__}")
print(f"Is CUDA available for bnb? {torch.cuda.is_available()}")

Bitsandbytes version: 0.49.2
Is CUDA available for bnb? True


In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from trl.chat_template_utils import qwen3_schema

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Load completely standard, unquantized (but in T4-friendly fp16)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
   torch_dtype=torch.float32 
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.response_schema = qwen3_schema # Fix the template bug

# 2. Clean, straightforward Config
grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=1e-5,            # Can use a slightly higher LR for full training
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    optim="adamw_torch",           # Standard optimizer works now!
    fp16=True,                     
    bf16=False,
    
    num_generations=2,
    max_completion_length=256,     
    
    gradient_checkpointing=True,
    use_vllm=False,                
    output_dir="thermal-ops-0.5B",
    report_to="none",              
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## Create Trainer


In [20]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=ThermalOpsEnv,
)

/tmp/ipykernel_17882/3703186129.py:1: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(


In [21]:
%pip install jmespath

## Memory Stats & Training Loop


In [ ]:
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss


## Evaluate Inference
Run the fine-tuned model against a generic Thermal-Ops environment test.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import get_peft_model
import json
import re

fine_tuned_model = AutoModelForCausalLM.from_pretrained(output_dir, torch_dtype="float32", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def play_thermal_ops(model, tokenizer):
    env = ThermalOpsEnv()
    
    # Introduce random difficulty (Hardware Failure Scenario)
    env.broken_fans.add(2)
    obs = env.reset()

    print("Initial observation:")
    print(obs)
    print()

    messages = [{"role": "user", "content": prompt}, {"role": "user", "content": obs}]

    for turn in range(env.max_steps * 5): # Maximum 5 tool calls per real step
        if env.done:
            break

        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )
        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=256)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

        print(f"Model output: {generated_text}")
        
        try:
            # Parse tool
            if "{" in generated_text and "}" in generated_text:
                start = generated_text.index("{")
                end = generated_text.rindex("}") + 1
                args = json.loads(generated_text[start:end])
                
                tool_name = "wait"
                if "set_fan_speed" in generated_text: tool_name = "set_fan_speed"
                elif "adjust_chiller" in generated_text: tool_name = "adjust_chiller"
                elif "migrate_workload" in generated_text: tool_name = "migrate_workload"

                # Execute
                if tool_name == "set_fan_speed":
                    feedback = env.set_fan_speed(args.get("rack_id", 0), args.get("rpm", 1000))
                elif tool_name == "adjust_chiller":
                    feedback = env.adjust_chiller(args.get("chiller_temp", 20.0))
                elif tool_name == "migrate_workload":
                    feedback = env.migrate_workload(args.get("source_rack", 0), args.get("target_rack", 1))
                else:
                    feedback = env.wait()
                    
                print(f"    Feedback: {feedback.strip()}")
                
                messages.append({"role": "assistant", "content": generated_text})
                messages.append({"role": "user", "content": feedback})
            else:
                feedback = env.wait()
                messages.append({"role": "assistant", "content": generated_text})
                messages.append({"role": "user", "content": "Format error. Skipped: " + feedback})
                
        except Exception as e:
            print(f"Error executing Action: {e}")
            break

    print(f"Game finished! Final reward computed sum: {env.reward}")

play_thermal_ops(fine_tuned_model, tokenizer)

